# EnglishHub: pronunciation scoring baseline

This Google Colab notebook trains a small supervised baseline using SpeechOcean762 and a frozen `facebook/wav2vec2-base` audio encoder. A trainable regression head predicts **sentence-level** scores for accuracy, completeness, fluency, prosody, and total.

**Scope limitation:** this notebook does not produce reliable per-word or phoneme scores and does not reproduce the reference UI yet. Although SpeechOcean762 includes word/phone labels, its Hugging Face rows do not provide word-level audio boundaries. Correct per-word feedback requires acoustic alignment or an aligned dataset; do not spread sentence scores across words and present them as word scores.

The dataset contains 5,000 English utterances from non-native speakers whose first language is Mandarin. This is useful for a prototype, but performance for Vietnamese learners is not established. Dataset page: [mispeech/speechocean762](https://huggingface.co/datasets/mispeech/speechocean762). Paper: [Zhang et al., Interspeech 2021](https://arxiv.org/abs/2104.01378). The Hugging Face dataset metadata declares Apache-2.0; verify the dataset card and citation before redistributing data or model artifacts.

In [ ]:
%pip -q install "datasets[audio]" transformers scikit-learn soundfile

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from datasets import Audio, concatenate_datasets, load_dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoFeatureExtractor, Wav2Vec2Model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODER_NAME = "facebook/wav2vec2-base"
SAMPLE_RATE = 16_000
TARGET_COLUMNS = ["accuracy", "completeness", "fluency", "prosodic", "total"]
print("Device:", DEVICE)

## Load and inspect the dataset

The published dataset provides official train/test splits. Here we combine them only to create a reproducible speaker-disjoint train/validation/test split, avoiding the same speaker appearing in multiple subsets.

In [ ]:
dataset = load_dataset("mispeech/speechocean762")
print(dataset)
print("Features:", dataset["train"].features)
print("Train rows:", len(dataset["train"]), "Test rows:", len(dataset["test"]))

dataset = {
    split: part.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    for split, part in dataset.items()
}
all_data = concatenate_datasets([dataset["train"], dataset["test"]])
example = all_data[0]
print("Example text:", example["text"])
print("Example speaker:", example["speaker"])
print("Sentence labels:", {name: example[name] for name in TARGET_COLUMNS})
print("Audio shape / rate:", np.asarray(example["audio"]["array"]).shape, example["audio"]["sampling_rate"])
print("Word annotation example:", example["words"][0])

## Create speaker-disjoint splits

The two-stage group split reserves roughly 15% for test and 15% for validation. The split is by speaker, not by individual recording.

In [ ]:
speaker_ids = np.asarray(all_data["speaker"])
row_ids = np.arange(len(all_data))

first_split = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val_idx, test_idx = next(first_split.split(row_ids, groups=speaker_ids))

second_split = GroupShuffleSplit(n_splits=1, test_size=0.1764705882, random_state=SEED)
train_rel_idx, val_rel_idx = next(
    second_split.split(train_val_idx, groups=speaker_ids[train_val_idx])
)
train_idx = train_val_idx[train_rel_idx]
val_idx = train_val_idx[val_rel_idx]

train_speakers = set(speaker_ids[train_idx])
val_speakers = set(speaker_ids[val_idx])
test_speakers = set(speaker_ids[test_idx])
assert train_speakers.isdisjoint(val_speakers)
assert train_speakers.isdisjoint(test_speakers)
assert val_speakers.isdisjoint(test_speakers)

print(f"Rows: train={len(train_idx)}, validation={len(val_idx)}, test={len(test_idx)}")
print(f"Speakers: train={len(train_speakers)}, validation={len(val_speakers)}, test={len(test_speakers)}")
print("Speaker overlap: none")

## Extract frozen Wav2Vec2 utterance embeddings

The audio encoder is pretrained and frozen. Mean pooling over valid audio frames gives one vector per utterance; the regression head below learns from those vectors. Feature extraction can take several minutes. Enable a Colab GPU via **Runtime → Change runtime type → T4 GPU** if available.

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(ENCODER_NAME)
encoder = Wav2Vec2Model.from_pretrained(ENCODER_NAME).to(DEVICE)
encoder.eval()
for parameter in encoder.parameters():
    parameter.requires_grad = False

def to_mono_float32(audio_item):
    waveform = np.asarray(audio_item["array"], dtype=np.float32)
    waveform = np.squeeze(waveform)
    if waveform.ndim == 2:
        channel_axis = 0 if waveform.shape[0] <= 2 else 1
        waveform = waveform.mean(axis=channel_axis)
    if waveform.ndim != 1 or waveform.size == 0:
        raise ValueError(f"Expected non-empty mono audio, got shape {waveform.shape}")
    if audio_item["sampling_rate"] != SAMPLE_RATE:
        raise ValueError(f"Expected {SAMPLE_RATE} Hz audio, got {audio_item['sampling_rate']} Hz")
    return waveform

@torch.inference_mode()
def extract_embeddings(rows, batch_size=8):
    results = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start : start + batch_size]
        waveforms = [to_mono_float32(row["audio"]) for row in batch]
        inputs = feature_extractor(
            waveforms,
            sampling_rate=SAMPLE_RATE,
            padding=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
        hidden = encoder(**inputs).last_hidden_state
        frame_mask = encoder._get_feature_vector_attention_mask(
            hidden.shape[1], inputs["attention_mask"]
        ).unsqueeze(-1)
        pooled = (hidden * frame_mask).sum(dim=1) / frame_mask.sum(dim=1).clamp(min=1)
        results.append(pooled.cpu().numpy().astype(np.float32))
        if (start // batch_size) % 50 == 0:
            print(f"Encoded {min(start + len(batch), len(rows))}/{len(rows)}")
    return np.concatenate(results, axis=0)

embeddings = extract_embeddings(all_data)
targets = np.asarray(
    [[float(row[name]) for name in TARGET_COLUMNS] for row in all_data],
    dtype=np.float32,
)
print("Embedding matrix:", embeddings.shape, "Target matrix:", targets.shape)

## Train the sentence-level score head

All five targets are on a 0–10 scale in this dataset and are normalized to 0–1 for training. The held-out test speakers are not used for model selection.

In [ ]:
X = torch.from_numpy(embeddings)
Y = torch.from_numpy(targets / 10.0)

train_loader = DataLoader(
    TensorDataset(X[train_idx], Y[train_idx]), batch_size=64, shuffle=True
)
val_x, val_y = X[val_idx].to(DEVICE), Y[val_idx].to(DEVICE)
test_x, test_y = X[test_idx].to(DEVICE), Y[test_idx].to(DEVICE)

class ScoreRegressor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, output_size),
            nn.Sigmoid(),
        )

    def forward(self, features):
        return self.layers(features)

head = ScoreRegressor(X.shape[1], len(TARGET_COLUMNS)).to(DEVICE)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.SmoothL1Loss()
best_val_mae = float("inf")
best_state = None
epochs_without_improvement = 0
PATIENCE = 7
MAX_EPOCHS = 60

for epoch in range(1, MAX_EPOCHS + 1):
    head.train()
    train_losses = []
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(head(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    head.eval()
    with torch.inference_mode():
        val_pred = head(val_x)
        val_mae = torch.mean(torch.abs(val_pred - val_y)).item()
    print(f"Epoch {epoch:02d}: train_loss={np.mean(train_losses):.4f}, val_MAE={val_mae * 10:.3f}/10")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_state = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("Training did not produce a validation checkpoint")
head.load_state_dict(best_state)
head.eval()

## Evaluate once on held-out speakers

Metrics are reported on the original 0–10 score scale. Do not treat these results as evidence of accuracy on Vietnamese learners; this test set comes from the same Mandarin-L1 corpus.

In [ ]:
with torch.inference_mode():
    test_predictions = (head(test_x).cpu().numpy() * 10.0).clip(0.0, 10.0)
test_truth = (test_y.cpu().numpy() * 10.0).clip(0.0, 10.0)

for index, name in enumerate(TARGET_COLUMNS):
    mae = mean_absolute_error(test_truth[:, index], test_predictions[:, index])
    rmse = np.sqrt(mean_squared_error(test_truth[:, index], test_predictions[:, index]))
    print(f"{name:12s} MAE={mae:.3f} / 10   RMSE={rmse:.3f} / 10")

In [ ]:
sample_index = int(test_idx[0])
print("Reference:", all_data[sample_index]["text"])
print("Human labels:", {name: round(float(targets[sample_index, i]), 2) for i, name in enumerate(TARGET_COLUMNS)})
print("Model scores:", {name: round(float(test_predictions[0, i]), 2) for i, name in enumerate(TARGET_COLUMNS)})

## Save artifacts and define inference helper

The checkpoint contains the regression head and the name of its frozen encoder. To run predictions elsewhere, load the same pretrained encoder and checkpoint. Keep the saved artifacts private until you have reviewed the dataset/model terms and your deployment requirements.

In [ ]:
OUTPUT_DIR = Path("/content/englishhub_pronunciation_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / "score_regressor.pt"
torch.save(
    {
        "head_state_dict": head.state_dict(),
        "embedding_size": int(X.shape[1]),
        "encoder_name": ENCODER_NAME,
        "sample_rate": SAMPLE_RATE,
        "target_columns": TARGET_COLUMNS,
    },
    checkpoint_path,
)

metadata = {
    "dataset": "mispeech/speechocean762",
    "encoder": ENCODER_NAME,
    "sample_rate": SAMPLE_RATE,
    "target_columns": TARGET_COLUMNS,
    "output_scale": "0-10",
    "scope": "sentence-level only; not word/phoneme pronunciation assessment",
    "limitations": ["dataset speakers have Mandarin as L1", "not validated for Vietnamese learners"],
    "test_speakers": sorted(test_speakers),
    "test_mae": {
        name: float(mean_absolute_error(test_truth[:, i], test_predictions[:, i]))
        for i, name in enumerate(TARGET_COLUMNS)
    },
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", checkpoint_path)
print("Saved:", OUTPUT_DIR / "metadata.json")

In [ ]:
@torch.inference_mode()
def predict_sentence_scores(waveform, sampling_rate=SAMPLE_RATE):
    """Predict sentence-level scores for one mono waveform; returns scores on a 0-10 scale."""
    if sampling_rate != SAMPLE_RATE:
        raise ValueError(f"Resample the waveform to {SAMPLE_RATE} Hz before calling this function")
    waveform = np.asarray(waveform, dtype=np.float32).squeeze()
    if waveform.ndim != 1 or waveform.size == 0:
        raise ValueError("waveform must be a non-empty mono audio array")
    inputs = feature_extractor(
        waveform,
        sampling_rate=SAMPLE_RATE,
        return_attention_mask=True,
        return_tensors="pt",
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    hidden = encoder(**inputs).last_hidden_state
    frame_mask = encoder._get_feature_vector_attention_mask(
        hidden.shape[1], inputs["attention_mask"]
    ).unsqueeze(-1)
    pooled = (hidden * frame_mask).sum(dim=1) / frame_mask.sum(dim=1).clamp(min=1)
    scores = (head(pooled).squeeze(0).cpu().numpy() * 10.0).clip(0.0, 10.0)
    return {name: float(score) for name, score in zip(TARGET_COLUMNS, scores)}

demo = all_data[sample_index]["audio"]
predict_sentence_scores(demo["array"], demo["sampling_rate"])

## Next step: per-word feedback

This model outputs sentence scores only. For the word coloring in the reference UI, the next experiment should use word/phoneme audio boundaries and corresponding labels (for example, investigate the manually aligned subset of L2-ARCTIC), or implement/validate forced alignment. Do not convert a sentence-level score into individual word scores. Also collect consented Vietnamese-L1 recordings and evaluate on speakers never seen during training before making claims about Vietnamese learners.